In [ ]:
# In[1]:

import pandas as pd

# Load CSVs and parse timestamps to UTC datetimes
metric = pd.read_csv("metric.csv")
trace = pd.read_csv("trace.csv")
log = pd.read_csv("log.csv")
error_logs = pd.read_csv("error_logs.csv")

for df in (metric, trace, log, error_logs):
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)

# Helper to extract compact time info
def time_bounds(s):
    if s.empty:
        return {"rows": 0, "earliest_unix": None, "earliest_iso": None, "latest_unix": None, "latest_iso": None}
    tmin = s.min()
    tmax = s.max()
    return {
        "rows": int(len(s)),
        "earliest_unix": int(tmin.timestamp()),
        "earliest_iso": tmin.isoformat(),
        "latest_unix": int(tmax.timestamp()),
        "latest_iso": tmax.isoformat(),
    }

# Metric inventory
metric_bounds = time_bounds(metric["timestamp"])
metric_cmdbs = metric["cmdb_id"].dropna().unique().tolist()[:50]
metric_kpis = metric["kpi_name"].dropna().unique().tolist()[:50]

# Top 20 (cmdb_id, kpi_name) pairs by row count
metric_pairs = (
    metric.groupby(["cmdb_id", "kpi_name"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .head(20)
)
metric_top_pairs = metric_pairs.to_dict(orient="records")

# Trace inventory
trace_bounds = time_bounds(trace["timestamp"])
trace_cmdbs = trace["cmdb_id"].dropna().unique().tolist()[:50]
trace_features = trace["trace_name"].dropna().unique().tolist()[:50]

# Log inventory
log_bounds = time_bounds(log["timestamp"])
log_cmdbs = log["cmdb_id"].dropna().unique().tolist()[:50]
log_features = log["log_name"].dropna().unique().tolist()[:50]

# Error logs inventory
error_bounds = time_bounds(error_logs["timestamp"])
error_cmdbs = error_logs["cmdb_id"].dropna().unique().tolist()[:50]

# Top 20 distinct message texts with counts and cmdb_id distribution
if not error_logs.empty:
    msg_counts = (
        error_logs.groupby(["message"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .head(20)
    )
    # Precompute message x cmdb_id counts
    msg_cmdb_counts = (
        error_logs.groupby(["message", "cmdb_id"])
        .size()
        .reset_index(name="count")
    )
    # Build list of top messages with distributions
    top_messages = []
    for _, row in msg_counts.iterrows():
        msg = row["message"]
        total = int(row["count"])
        d = msg_cmdb_counts[msg_cmdb_counts["message"] == msg][["cmdb_id", "count"]]
        # Convert to dict of cmdb_id -> int count
        cmdb_dist = {str(r["cmdb_id"]): int(r["count"]) for _, r in d.iterrows()}
        top_messages.append({"message": msg, "count": total, "cmdb_distribution": cmdb_dist})
else:
    top_messages = []

# Compose final JSON-serializable inventory
inventory = {
    "metric": {
        "summary": metric_bounds,
        "unique_cmdb_id_sample": metric_cmdbs,
        "unique_kpi_sample": metric_kpis,
        "top_20_cmdb_kpi_pairs": metric_top_pairs,
    },
    "trace": {
        "summary": trace_bounds,
        "unique_cmdb_id_sample": trace_cmdbs,
        "unique_trace_feature_sample": trace_features,
    },
    "log": {
        "summary": log_bounds,
        "unique_cmdb_id_sample": log_cmdbs,
        "unique_log_feature_sample": log_features,
    },
    "error_logs": {
        "summary": error_bounds,
        "unique_cmdb_id_sample": error_cmdbs,
        "top_20_messages": top_messages,
    },
}

inventory

```
Out[1]:
```
```python
summary = (
    "Inventory summary:\n"
    "- metric.csv: 1925 rows from 1705732680 (2024-01-20T06:38:00+00:00) to 1705734120 (2024-01-20T07:02:00+00:00). "
    "Sampled services: carts, carts-db, catalogue, catalogue-db, front-end, orders, orders-db, payment, queue-master, rabbitmq, rabbitmq-exporter, session-db, shipping, user, user-db. "
    "Top KPI names sampled include: cpu, diskio, latency-50, latency-90, mem, socket, workload, error. "
    "Top (cmdb_id, kpi_name) pairs are dominated by many 25-row series (e.g., carts, carts-db, catalogue, catalogue-db and others).\n"
    "- trace.csv: 0 rows (no trace data available).\n"
    "- log.csv: 452 rows from 1705732680 (2024-01-20T06:38:00+00:00) to 1705734120 (2024-01-20T07:02:00+00:00). "
    "Sampled services include carts, catalogue, front-end, orders, payment, queue-master, shipping, user, carts-db, session-db. "
    "Log features present: log.error_count, log.row_count.\n"
    "- error_logs.csv: 0 rows (no error log messages).\n\n"
    "Overall: metric and log telemetry exist for the ~24-minute window shown; trace and error-log data are absent. "
    "Metrics cover core Sock Shop services (carts, catalogue, orders, payment, user, DBs, front-end); KPI coverage is broad (CPU, memory, disk I/O, latencies, sockets, workload)."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

{'metric': {'summary': {'rows': 1925, 'earliest_unix': 1705732680, 'earliest_iso': '2024-01-20T06:38:00+00:00', 'latest_unix': 1705734120, 'latest_iso': '2024-01-20T07:02:00+00:00'}, 'unique_cmdb_id_sample': ['carts', 'carts-db', 'catalogue', 'catalogue-db', 'front-end', 'orders', 'orders-db', 'payment', 'queue-master', 'rabbitmq', 'rabbitmq-exporter', 'session-db', 'shipping', 'user', 'user-db'], 'unique_kpi_sample': ['cpu', 'diskio', 'latency-50', 'latency-90', 'mem', 'socket', 'workload', 'error'], 'top_20_cmdb_kpi_pairs': [{'cmdb_id': 'carts', 'kpi_name': 'cpu', 'count': 25}, {'cmdb_id': 'carts', 'kpi_name': 'diskio', 'count': 25}, {'cmdb_id': 'carts', 'kpi_name': 'latency-50', 'count': 25}, {'cmdb_id': 'carts', 'kpi_name': 'latency-90', 'count': 25}, {'cmdb_id': 'carts', 'kpi_name': 'mem', 'count': 25}, {'cmdb_id': 'carts', 'kpi_name': 'socket', 'count': 25}, {'cmdb_id': 'carts', 'kpi_name': 'workload', 'count': 25}, {'cmdb_id': 'carts-db', 'kpi_name': 'cpu', 'count': 25}, {'cmdb_id': 'carts-db', 'kpi_name': 'diskio', 'count': 25}, {'cmdb_id': 'carts-db', 'kpi_name': 'mem', 'count': 25}, {'cmdb_id': 'carts-db', 'kpi_name': 'socket', 'count': 25}, {'cmdb_id': 'catalogue', 'kpi_name': 'cpu', 'count': 25}, {'cmdb_id': 'catalogue', 'kpi_name': 'latency-50', 'count': 25}, {'cmdb_id': 'catalogue', 'kpi_name': 'latency-90', 'count': 25}, {'cmdb_id': 'catalogue', 'kpi_name': 'mem', 'count': 25}, {'cmdb_id': 'catalogue', 'kpi_name': 'socket', 'count': 25}, {'cmdb_id': 'catalogue', 'kpi_name': 'workload', 'count': 25}, {'cmdb_id': 'catalogue-db', 'kpi_name': 'cpu', 'count': 25}, {'cmdb_id': 'catalogue-db', 'kpi_name': 'diskio', 'count': 25}, {'cmdb_id': 'catalogue-db', 'kpi_name': 'mem', 'count': 25}]}, 'trace': {'summary': {'rows': 0, 'earliest_unix': None, 'earliest_iso': None, 'latest_unix': None, 'latest_iso': None}, 'unique_cmdb_id_sample': [], 'unique_trace_feature_sample': []}, 'log': {'summary': {'rows': 452, 'earliest_unix': 1705732680, 'earliest_iso': '2024-01-20T06:38:00+00:00', 'latest_unix': 1705734120, 'latest_iso': '2024-01-20T07:02:00+00:00'}, 'unique_cmdb_id_sample': ['carts', 'catalogue', 'front-end', 'orders', 'payment', 'queue-master', 'shipping', 'user', 'carts-db', 'session-db'], 'unique_log_feature_sample': ['log.error_count', 'log.row_count']}, 'error_logs': {'summary': {'rows': 0, 'earliest_unix': None, 'earliest_iso': None, 'latest_unix': None, 'latest_iso': None}, 'unique_cmdb_id_sample': [], 'top_20_messages': []}}```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse existing metric variable if present; otherwise load from CSV
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")

# 1) Ensure timestamp parsed to UTC datetimes
metric["timestamp"] = pd.to_datetime(metric["timestamp"], unit="s", utc=True)

# 2) Compute global P95 for each (cmdb_id, kpi_name) pair using the full series (before any filtering)
group_cols = ["cmdb_id", "kpi_name"]
# total rows per series (full)
series_counts = metric.groupby(group_cols).size().reset_index(name="total_rows_full")
# global P95 per series
p95_df = metric.groupby(group_cols)["value"].quantile(0.95).reset_index(name="p95")

# Merge counts and p95 into one DataFrame
series_meta = series_counts.merge(p95_df, on=group_cols)

# 3) Filter metric.csv to the incident window (inclusive)
start = pd.to_datetime("2024-01-20T06:35:37Z", utc=True)
end = pd.to_datetime("2024-01-20T07:05:37Z", utc=True)
metric_window = metric[(metric["timestamp"] >= start) & (metric["timestamp"] <= end)].copy()

# 4) For each pair, find incident-window points that exceed the global P95 (value > P95)
# Join p95 onto the windowed points
metric_window = metric_window.merge(p95_df, on=group_cols, how="left")

# Mark anomalies: value > p95 (strict)
metric_window["is_anomaly"] = metric_window["value"] > metric_window["p95"]

# Aggregate anomaly info per series
anoms_agg = (
    metric_window[metric_window["is_anomaly"]]
    .groupby(group_cols)
    .agg(
        anomaly_count=("is_anomaly", "size"),
        earliest_anom_ts=("timestamp", "min"),
        max_anom_value=("value", "max"),
    )
    .reset_index()
)

# Convert earliest timestamp to unix and ISO strings
if not anoms_agg.empty:
    anoms_agg["earliest_anom_unix"] = anoms_agg["earliest_anom_ts"].apply(lambda t: int(t.timestamp()))
    anoms_agg["earliest_anom_iso"] = anoms_agg["earliest_anom_ts"].apply(lambda t: t.isoformat())
else:
    anoms_agg["earliest_anom_unix"] = pd.Series(dtype=int)
    anoms_agg["earliest_anom_iso"] = pd.Series(dtype=object)

# 5) Prepare final ranked list with required fields
result = (
    anoms_agg.merge(series_meta, on=group_cols, how="left")
    .loc[:, ["cmdb_id", "kpi_name", "total_rows_full", "p95", "anomaly_count", "earliest_anom_unix", "earliest_anom_iso", "max_anom_value"]]
    .sort_values(["anomaly_count", "max_anom_value"], ascending=[False, False])
    .reset_index(drop=True)
)

# Limit to top 20
top20_anomalies = result.head(20)

# Small summary
total_series_checked = int(series_meta.shape[0])
total_series_with_anomaly = int(result.shape[0])
summary = {
    "total_series_checked": total_series_checked,
    "total_series_with_anomaly": total_series_with_anomaly,
}

# Display compact results: top20_anomalies (DataFrame) and summary (dict)
top20_anomalies, summary

```
Out[2]:
```
summary_text = (
    "Summary of anomaly scan (metric.csv) for window 2024-01-20T06:35:37Z to 2024-01-20T07:05:37Z:\n"
    f"- Total distinct (cmdb_id, kpi_name) series checked: {summary['total_series_checked']}\n"
    f"- Series with >=1 point above their global P95 in the window: {summary['total_series_with_anomaly']}\n\n"
    "Top anomalous series (ranked by number of anomaly points, up to 10 shown):\n"
)
# Use top20_anomalies variable produced earlier; show up to 10 rows
rows = top20_anomalies.head(10)
for i, r in rows.iterrows():
    summary_text += (
        f"{i+1}. {r['cmdb_id']} | {r['kpi_name']}: total_rows={int(r['total_rows_full'])}, "
        f"P95={r['p95']:.6g}, anomalies={int(r['anomaly_count'])}, "
        f"earliest_anom={int(r['earliest_anom_unix'])} ({r['earliest_anom_iso']}), "
        f"max_anom_value={r['max_anom_value']:.6g}\n"
    )

summary_text += (
    "\nOverall observation: anomalies are widespread (61/77 series) and concentrated in memory (mem) "
    "and disk I/O (diskio) KPIs across many services (e.g., catalogue-db, orders, shipping, queue-master, carts, front-end)."
)

summary_text

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id kpi_name  total_rows_full           p95  anomaly_count  earliest_anom_unix          earliest_anom_iso  max_anom_value
0   catalogue-db      mem               25  4.306893e+08              2          1705733940  2024-01-20T06:59:00+00:00    4.307050e+08
1         orders      mem               25  3.373224e+08              2          1705733940  2024-01-20T06:59:00+00:00    3.373418e+08
2       shipping      mem               25  3.038169e+08              2          1705733940  2024-01-20T06:59:00+00:00    3.038228e+08
3   queue-master      mem               25  2.487030e+08              2          1705733280  2024-01-20T06:48:00+00:00    2.488939e+08
4          carts      mem               25  2.208497e+08              2          1705733640  2024-01-20T06:54:00+00:00    2.208755e+08
5      front-end      mem               25  1.065184e+08              2          1705732680  2024-01-20T06:38:00+00:00    1.079182e+08
6       rabbitmq      mem               25  9.708167e+07              2          1705734060  2024-01-20T07:01:00+00:00    9.712258e+07
7       carts-db      mem               25  9.472927e+07              2          1705734060  2024-01-20T07:01:00+00:00    9.487371e+07
8        user-db      mem               25  5.819264e+07              2          1705734060  2024-01-20T07:01:00+00:00    5.845197e+07
9      orders-db      mem               25  4.699193e+07              2          1705734060  2024-01-20T07:01:00+00:00    4.704288e+07
10          user      mem               25  8.266998e+06              2          1705734060  2024-01-20T07:01:00+00:00    8.277693e+06
11     catalogue      mem               25  6.273069e+06              2          1705734000  2024-01-20T07:00:00+00:00    6.276164e+06
12       payment      mem               25  4.605051e+06              2          1705733940  2024-01-20T06:59:00+00:00    4.606976e+06
13    session-db      mem               25  2.918932e+06              2          1705733220  2024-01-20T06:47:00+00:00    2.929391e+06
14      carts-db   diskio               25  2.417570e+06              2          1705732680  2024-01-20T06:38:00+00:00    2.478130e+06
15       user-db   diskio               25  1.657373e+06              2          1705733640  2024-01-20T06:54:00+00:00    1.719453e+06
16     orders-db   diskio               25  8.728126e+05              2          1705733280  2024-01-20T06:48:00+00:00    8.931707e+05
17        orders   diskio               25  9.792931e+04              2          1705732680  2024-01-20T06:38:00+00:00    1.533240e+05
18  queue-master   diskio               25  5.435826e+04              2          1705734000  2024-01-20T07:00:00+00:00    1.293541e+05
19    session-db   diskio               25  1.233237e+04              2          1705732800  2024-01-20T06:40:00+00:00    1.565403e+04, {'total_series_checked': 77, 'total_series_with_anomaly': 61})```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse metric if present; otherwise load
try:
    metric
except NameError:
    metric = pd.read_csv("metric.csv")

# 1) Parse timestamp to UTC datetimes
metric["timestamp"] = pd.to_datetime(metric["timestamp"], unit="s", utc=True)

# 2) Compute global P95 per (cmdb_id, kpi_name) using full series (global thresholds BEFORE filtering)
group_cols = ["cmdb_id", "kpi_name"]
p95_df = metric.groupby(group_cols)["value"].quantile(0.95).reset_index(name="p95")

# 3) Filter to incident window (inclusive)
start = pd.to_datetime("2024-01-20T06:35:37Z", utc=True)
end = pd.to_datetime("2024-01-20T07:05:37Z", utc=True)
metric_window = metric[(metric["timestamp"] >= start) & (metric["timestamp"] <= end)].copy()

# 4) Mark points where value > global P95 as anomalous
metric_window = metric_window.merge(p95_df, on=group_cols, how="left")
metric_window["is_anomaly"] = metric_window["value"] > metric_window["p95"]

# Keep only anomalous points for segmentation
anoms = metric_window[metric_window["is_anomaly"]].copy()
anoms = anoms.sort_values(group_cols + ["timestamp"]).reset_index(drop=True)

# If no anomalies, prepare empty outputs
if anoms.empty:
    top_segments = pd.DataFrame(columns=[
        "cmdb_id","kpi_name","segment_start_unix","segment_start_iso",
        "segment_end_unix","segment_end_iso","length_minutes",
        "segment_max_value","global_p95","breach_ratio"
    ])
    per_cmdb_agg = pd.DataFrame(columns=["cmdb_id","number_of_fault_segments","affected_kpis_counts","max_breach_ratio"])
else:
    # 4-5) Group anomalous points into consecutive fault segments where successive timestamps differ by exactly 60s
    def assign_segment_ids(df):
        df = df.sort_values("timestamp").copy()
        # compute diff in seconds
        df["diff_s"] = df["timestamp"].diff().dt.total_seconds().fillna(np.nan)
        # start new segment when diff != 60
        df["new_seg"] = (df["diff_s"] != 60).astype(int)
        # cumulative sum of new_seg gives segment ids starting at 1 within this group
        df["seg_id_within_group"] = df["new_seg"].cumsum()
        return df

    anoms = anoms.groupby(group_cols, group_keys=False).apply(assign_segment_ids).reset_index(drop=True)

    # Build segment-level aggregation
    seg_group_cols = group_cols + ["seg_id_within_group"]
    segs = (
        anoms.groupby(seg_group_cols)
        .agg(
            segment_start_ts=("timestamp", "min"),
            segment_end_ts=("timestamp", "max"),
            length_points=("timestamp", "size"),
            segment_max_value=("value", "max"),
            global_p95=("p95", "first"),
        )
        .reset_index()
    )

    # 5) Keep only segments with length >= 2 (consecutive anomalies)
    segs = segs[segs["length_points"] >= 2].copy()

    # If none remain after filtering
    if segs.empty:
        top_segments = pd.DataFrame(columns=[
            "cmdb_id","kpi_name","segment_start_unix","segment_start_iso",
            "segment_end_unix","segment_end_iso","length_minutes",
            "segment_max_value","global_p95","breach_ratio"
        ])
        per_cmdb_agg = pd.DataFrame(columns=["cmdb_id","number_of_fault_segments","affected_kpis_counts","max_breach_ratio"])
    else:
        # 6) Compute required fields
        segs["segment_start_unix"] = segs["segment_start_ts"].apply(lambda t: int(t.timestamp()))
        segs["segment_start_iso"] = segs["segment_start_ts"].apply(lambda t: t.isoformat())
        segs["segment_end_unix"] = segs["segment_end_ts"].apply(lambda t: int(t.timestamp()))
        segs["segment_end_iso"] = segs["segment_end_ts"].apply(lambda t: t.isoformat())
        # Length in minutes: consecutive points spaced by 60s -> length_minutes = number of points
        segs["length_minutes"] = segs["length_points"]
        segs["segment_max_value"] = segs["segment_max_value"].astype(float)
        segs["global_p95"] = segs["global_p95"].astype(float)
        # breach_ratio = (segment_max_value - p95)/p95 ; handle p95 == 0 safely
        def compute_breach(row):
            p = row["global_p95"]
            mv = row["segment_max_value"]
            if p == 0:
                if mv == 0:
                    return 0.0
                else:
                    return np.inf
            return (mv - p) / p
        segs["breach_ratio"] = segs.apply(compute_breach, axis=1)

        # Select and order columns
        segs_out = segs[[
            "cmdb_id","kpi_name","segment_start_unix","segment_start_iso",
            "segment_end_unix","segment_end_iso","length_minutes",
            "segment_max_value","global_p95","breach_ratio"
        ]].copy()

        # 7) Aggregate per-cmdb_id summary: number_of_fault_segments, list of affected_kpis (count each), max_breach_ratio
        agg_kpis = (
            segs_out.groupby(["cmdb_id","kpi_name"]).size().reset_index(name="kpi_segment_count")
        )
        # build affected_kpis_counts per cmdb_id as dict-like string to keep compact
        per_cmdb = (
            agg_kpis.groupby("cmdb_id").apply(
                lambda df: pd.Series({
                    "number_of_fault_segments": int(df["kpi_segment_count"].sum()),
                    "affected_kpis_counts": dict(zip(df["kpi_name"], df["kpi_segment_count"]))
                })
            ).reset_index()
        )

        # compute max breach ratio per cmdb_id
        max_breach = segs_out.groupby("cmdb_id")["breach_ratio"].max().reset_index(name="max_breach_ratio")
        per_cmdb_agg = per_cmdb.merge(max_breach, on="cmdb_id", how="left")

        # 8) Limit segments to top 20 by breach_ratio descending
        top_segments = segs_out.sort_values("breach_ratio", ascending=False).head(20).reset_index(drop=True)

# Final compact outputs: top_segments (top 20 segments) and per_cmdb_agg (full aggregate)
top_segments, per_cmdb_agg

```
Out[3]:
```
fault_summary = (
    "Fault identification summary for 2024-01-20T06:35:37Z to 2024-01-20T07:05:37Z:\n\n"
    "Overall:\n"
    "- Total fault segments (consecutive anomalous points, length>=2): 23\n"
    "- Segments retained were across many KPIs; common KPI categories: mem, diskio, cpu, latency-50/90, socket.\n\n"
    "Top fault segments (by breach ratio, top 5):\n"
    "1) queue-master | diskio — 2024-01-20T07:00:00 to 2024-01-20T07:01:00 (2 min): max=129,354.1, p95=54,358.26, breach_ratio=1.3797\n"
    "2) queue-master | cpu    — 2024-01-20T06:47:00 to 2024-01-20T06:48:00 (2 min): max=2.050078, p95=1.307498, breach_ratio=0.5679\n"
    "3) orders      | diskio — 2024-01-20T06:38:00 to 2024-01-20T06:39:00 (2 min): max=153,324.0, p95=97,929.31, breach_ratio=0.5657\n"
    "4) shipping    | latency-90 — 2024-01-20T06:39:00 to 2024-01-20T06:40:00 (2 min): max=0.0091894, p95=0.0060435, breach_ratio=0.5205\n"
    "5) shipping    | cpu    — 2024-01-20T06:47:00 to 2024-01-20T06:48:00 (2 min): max=1.995688, p95=1.571402, breach_ratio=0.2700\n\n"
    "Notable per-service aggregates (number_of_fault_segments, max_breach_ratio):\n"
    "- shipping: 4 segments, max_breach_ratio=0.520546\n"
    "- queue-master: 3 segments, max_breach_ratio=1.379659\n"
    "- orders: 3 segments, max_breach_ratio=0.565660\n"
    "- user: 3 segments, max_breach_ratio=0.021964\n"
    "- carts: 2 segments, max_breach_ratio=0.223417\n"
    "- carts-db: 2 segments, max_breach_ratio=0.060525\n"
    "- catalogue: 1 segment, max_breach_ratio=0.000493\n"
    "- front-end: 1 segment, max_breach_ratio=0.006483\n"
    "- payment: 1 segment, max_breach_ratio=0.005759\n"
    "- orders-db: 1 segment, max_breach_ratio=0.001084\n"
    "- rabbitmq: 1 segment, max_breach_ratio=0.000421\n"
    "- user-db: 1 segment, max_breach_ratio=0.004456\n\n"
    "Conclusions / next steps:\n"
    "- The most severe breach is queue-master (diskio) with breach_ratio > 1 (i.e., segment max > 2x its global P95). Investigate queue-master disk I/O usage and recent workload/queue activity.\n"
    "- Orders and shipping also show high breach ratios (notably orders disk I/O and shipping latency), so check downstream dependencies and whether queue-master I/O issues could impact them.\n"
    "- Many memory (mem) anomalies exist but with small breach ratios; prioritize high breach_ratio items (diskio, cpu, latency) first.\n"
)
fault_summary

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id    kpi_name  segment_start_unix          segment_start_iso  segment_end_unix            segment_end_iso  length_minutes  segment_max_value    global_p95  breach_ratio
0   queue-master      diskio          1705734000  2024-01-20T07:00:00+00:00        1705734060  2024-01-20T07:01:00+00:00               2       1.293541e+05  5.435826e+04      1.379659
1   queue-master         cpu          1705733220  2024-01-20T06:47:00+00:00        1705733280  2024-01-20T06:48:00+00:00               2       2.050078e+00  1.307498e+00      0.567939
2         orders      diskio          1705732680  2024-01-20T06:38:00+00:00        1705732740  2024-01-20T06:39:00+00:00               2       1.533240e+05  9.792931e+04      0.565660
3       shipping  latency-90          1705732740  2024-01-20T06:39:00+00:00        1705732800  2024-01-20T06:40:00+00:00               2       9.189404e-03  6.043489e-03      0.520546
4       shipping         cpu          1705733220  2024-01-20T06:47:00+00:00        1705733280  2024-01-20T06:48:00+00:00               2       1.995688e+00  1.571402e+00      0.270005
5          carts         cpu          1705733220  2024-01-20T06:47:00+00:00        1705733280  2024-01-20T06:48:00+00:00               2       2.563990e+00  2.095762e+00      0.223417
6       carts-db         cpu          1705734000  2024-01-20T07:00:00+00:00        1705734060  2024-01-20T07:01:00+00:00               2       6.990006e+00  6.591079e+00      0.060525
7           user  latency-90          1705734000  2024-01-20T07:00:00+00:00        1705734060  2024-01-20T07:01:00+00:00               2       6.986487e-03  6.836332e-03      0.021964
8       shipping      socket          1705732740  2024-01-20T06:39:00+00:00        1705732800  2024-01-20T06:40:00+00:00               2       6.300000e+00  6.213333e+00      0.013948
9          carts  latency-50          1705734000  2024-01-20T07:00:00+00:00        1705734060  2024-01-20T07:01:00+00:00               2       2.091232e-02  2.073228e-02      0.008684
10          user  latency-50          1705734000  2024-01-20T07:00:00+00:00        1705734060  2024-01-20T07:01:00+00:00               2       3.401013e-03  3.375363e-03      0.007599
11     front-end  latency-50          1705734000  2024-01-20T07:00:00+00:00        1705734060  2024-01-20T07:01:00+00:00               2       3.542199e-02  3.519383e-02      0.006483
12        orders         cpu          1705733880  2024-01-20T06:58:00+00:00        1705733940  2024-01-20T06:59:00+00:00               2       7.774146e+01  7.728787e+01      0.005869
13       payment  latency-50          1705733520  2024-01-20T06:52:00+00:00        1705733580  2024-01-20T06:53:00+00:00               2       2.950386e-03  2.933493e-03      0.005759
14       user-db         mem          1705734060  2024-01-20T07:01:00+00:00        1705734120  2024-01-20T07:02:00+00:00               2       5.845197e+07  5.819264e+07      0.004456
15      carts-db         mem          1705734060  2024-01-20T07:01:00+00:00        1705734120  2024-01-20T07:02:00+00:00               2       9.487371e+07  9.472927e+07      0.001525
16          user         mem          1705734060  2024-01-20T07:01:00+00:00        1705734120  2024-01-20T07:02:00+00:00               2       8.277693e+06  8.266998e+06      0.001294
17     orders-db         mem          1705734060  2024-01-20T07:01:00+00:00        1705734120  2024-01-20T07:02:00+00:00               2       4.704288e+07  4.699193e+07      0.001084
18  queue-master         mem          1705733280  2024-01-20T06:48:00+00:00        1705733340  2024-01-20T06:49:00+00:00               2       2.488939e+08  2.487030e+08      0.000768
19     catalogue         mem          1705734000  2024-01-20T07:00:00+00:00        1705734060  2024-01-20T07:01:00+00:00               2       6.276164e+06  6.273069e+06      0.000493,          cmdb_id  number_of_fault_segments                               affected_kpis_counts  max_breach_ratio
0          carts                         2                        {'cpu': 1, 'latency-50': 1}          0.223417
1       carts-db                         2                               {'cpu': 1, 'mem': 1}          0.060525
2      catalogue                         1                                         {'mem': 1}          0.000493
3      front-end                         1                                  {'latency-50': 1}          0.006483
4         orders                         3                  {'cpu': 1, 'diskio': 1, 'mem': 1}          0.565660
5      orders-db                         1                                         {'mem': 1}          0.001084
6        payment                         1                                  {'latency-50': 1}          0.005759
7   queue-master                         3                  {'cpu': 1, 'diskio': 1, 'mem': 1}          1.379659
8       rabbitmq                         1                                         {'mem': 1}          0.000421
9       shipping                         4  {'cpu': 1, 'latency-90': 1, 'mem': 1, 'socket'...          0.520546
10          user                         3       {'latency-50': 1, 'latency-90': 1, 'mem': 1}          0.021964
11       user-db                         1                                         {'mem': 1}          0.004456)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing 'log' if available, otherwise load
try:
    log
except NameError:
    log = pd.read_csv("log.csv")

# 1) Parse timestamp to UTC datetimes
log["timestamp"] = pd.to_datetime(log["timestamp"], unit="s", utc=True)

# Define grouping columns
group_cols = ["cmdb_id", "log_name"]

# 2) Compute global P95 and P5 using full series (before filtering)
agg_p = log.groupby(group_cols)["value"].quantile([0.95, 0.05]).unstack(level=-1)
# After unstack, columns might be 0.95 and 0.05; rename for clarity
agg_p = agg_p.rename(columns={0.95: "p95", 0.05: "p5"}).reset_index()

# Also compute total rows per series (full)
counts = log.groupby(group_cols).size().reset_index(name="total_rows_full")
series_meta = counts.merge(agg_p, on=group_cols, how="left")

# 3) Filter to incident window (inclusive)
start = pd.to_datetime("2024-01-20T06:35:37Z", utc=True)
end = pd.to_datetime("2024-01-20T07:05:37Z", utc=True)
log_window = log[(log["timestamp"] >= start) & (log["timestamp"] <= end)].copy()

# 4) Identify anomalous points per rules:
# A) log.error_count where value > P95
# B) log.row_count where value <= P5
# Merge global thresholds onto windowed data
log_window = log_window.merge(series_meta, on=group_cols, how="left")

def is_anom(row):
    if row["log_name"] == "log.error_count":
        return row["value"] > row["p95"]
    if row["log_name"] == "log.row_count":
        # anomaly when unusually low
        return row["value"] <= row["p5"]
    return False

log_window["is_anomaly"] = log_window.apply(is_anom, axis=1)

# Keep only anomalous points
anoms = log_window[log_window["is_anomaly"]].copy()

# 5) For each (cmdb_id, log_name) that has >=1 anomalous point, compute required summary fields
if anoms.empty:
    anomalous_summary = pd.DataFrame(columns=[
        "cmdb_id","log_name","total_rows_full","global_p95","global_p5",
        "anomaly_count_in_window","earliest_anom_unix","earliest_anom_iso",
        "extreme_anom_value","note"
    ])
else:
    # Compute per-series anomaly aggregates
    def extreme_agg(df):
        ln = df["log_name"].iat[0]
        if ln == "log.error_count":
            extreme_val = df["value"].max()
        else:  # log.row_count
            extreme_val = df["value"].min()
        return pd.Series({
            "anomaly_count_in_window": int(df.shape[0]),
            "earliest_anom_ts": df["timestamp"].min(),
            "extreme_anom_value": float(extreme_val)
        })

    anom_agg = (
        anoms.groupby(group_cols)
        .apply(extreme_agg)
        .reset_index()
    )

    # Merge with series_meta to get totals and global thresholds
    anomalous_summary = anom_agg.merge(series_meta, on=group_cols, how="left")

    # Format fields
    anomalous_summary["earliest_anom_unix"] = anomalous_summary["earliest_anom_ts"].apply(lambda t: int(t.timestamp()))
    anomalous_summary["earliest_anom_iso"] = anomalous_summary["earliest_anom_ts"].apply(lambda t: t.isoformat())
    anomalous_summary = anomalous_summary.rename(columns={
        "p95": "global_p95",
        "p5": "global_p5",
        "extreme_anom_value": "extreme_anom_value"
    })
    # Add note if cmdb_id == 'queue-master'
    anomalous_summary["note"] = anomalous_summary["cmdb_id"].apply(lambda x: "queue-master" if x == "queue-master" else "")

    # Select and order required columns; for error_count show max, for row_count show min in extreme_anom_value column
    anomalous_summary = anomalous_summary[[
        "cmdb_id","log_name","total_rows_full","global_p95","global_p5",
        "anomaly_count_in_window","earliest_anom_unix","earliest_anom_iso",
        "extreme_anom_value","note"
    ]]

# 6) Limit to top 20 series by anomaly_count descending
if not anomalous_summary.empty:
    top_anomalous_logs = anomalous_summary.sort_values("anomaly_count_in_window", ascending=False).head(20).reset_index(drop=True)
else:
    top_anomalous_logs = anomalous_summary

# 7) Small summary: total distinct series checked and how many had anomalies
total_series_checked = int(series_meta.shape[0])
total_series_with_anomalies = int(anomalous_summary.shape[0])

summary = {
    "total_distinct_series_checked": total_series_checked,
    "total_series_with_anomalies": total_series_with_anomalies
}

# Display compact results
top_anomalous_logs, summary

```
Out[4]:
```
Summary (log.csv) for 2024-01-20T06:35:37Z — 2024-01-20T07:05:37Z

- Dataset checked: 20 distinct (cmdb_id, log_name) series.  
- Series with anomalies in the incident window: 10.

Top anomalous series (ordered by anomaly count):
1. carts-db | log.row_count — total_rows_full=24, p95=1.00, p5=1.00, anomaly_count=24, earliest=2024-01-20T06:39:00Z, extreme_value=1.0  
2. carts    | log.row_count — total_rows_full=22, p95=123.75, p5=25.00, anomaly_count=8,  earliest=2024-01-20T06:38:00Z, extreme_value=25.0  
3. session-db | log.row_count — total_rows_full=5, p95=5.00, p5=5.00, anomaly_count=5, earliest=2024-01-20T06:40:00Z, extreme_value=5.0  
4. catalogue | log.row_count — total_rows_full=25, p95=173.40, p5=111.20, anomaly_count=2, earliest=2024-01-20T06:38:00Z, extreme_value=59.0  
5. orders   | log.row_count — total_rows_full=25, p95=135.20, p5=83.20, anomaly_count=2, earliest=2024-01-20T06:38:00Z, extreme_value=44.0  
6. front-end | log.row_count — total_rows_full=25, p95=2080.80, p5=1295.60, anomaly_count=2, earliest=2024-01-20T06:38:00Z, extreme_value=745.0  
7. payment  | log.row_count — total_rows_full=25, p95=107.60, p5=68.80, anomaly_count=2, earliest=2024-01-20T06:38:00Z, extreme_value=38.0  
8. queue-master | log.row_count — total_rows_full=25, p95=608.40, p5=374.40, anomaly_count=2, earliest=2024-01-20T06:38:00Z, extreme_value=198.0  (note: queue-master)
9. shipping | log.row_count — total_rows_full=25, p95=67.60, p5=41.60, anomaly_count=2, earliest=2024-01-20T06:38:00Z, extreme_value=22.0  
10. user    | log.row_count — total_rows_full=25, p95=632.60, p5=397.00, anomaly_count=2, earliest=2024-01-20T06:38:00Z, extreme_value=216.0

Key observations and interpretation:
- All detected anomalies are in log.row_count (low values at or below the global P5). No log.error_count series exceeded its P95 in the window.  
- Several services show sustained low row_count (e.g., carts-db shows value=1.0 for 24 points), indicating reduced log volume or traffic for those services during the window.  
- queue-master is flagged (row_count anomalies) — worth checking queue-master ingestion/throughput and whether it could affect downstream services.  
- Actionable next steps: investigate whether low log.row_count corresponds to lower traffic, a logging/ingestion issue, or a failure in components emitting logs (start with carts-db, carts, session-db, and queue-master).

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id       log_name  total_rows_full  global_p95  global_p5  anomaly_count_in_window  earliest_anom_unix          earliest_anom_iso  extreme_anom_value          note
0      carts-db  log.row_count               24        1.00        1.0                       24          1705732740  2024-01-20T06:39:00+00:00                 1.0              
1         carts  log.row_count               22      123.75       25.0                        8          1705732680  2024-01-20T06:38:00+00:00                25.0              
2    session-db  log.row_count                5        5.00        5.0                        5          1705732800  2024-01-20T06:40:00+00:00                 5.0              
3     catalogue  log.row_count               25      173.40      111.2                        2          1705732680  2024-01-20T06:38:00+00:00                59.0              
4        orders  log.row_count               25      135.20       83.2                        2          1705732680  2024-01-20T06:38:00+00:00                44.0              
5     front-end  log.row_count               25     2080.80     1295.6                        2          1705732680  2024-01-20T06:38:00+00:00               745.0              
6       payment  log.row_count               25      107.60       68.8                        2          1705732680  2024-01-20T06:38:00+00:00                38.0              
7  queue-master  log.row_count               25      608.40      374.4                        2          1705732680  2024-01-20T06:38:00+00:00               198.0  queue-master
8      shipping  log.row_count               25       67.60       41.6                        2          1705732680  2024-01-20T06:38:00+00:00                22.0              
9          user  log.row_count               25      632.60      397.0                        2          1705732680  2024-01-20T06:38:00+00:00               216.0              , {'total_distinct_series_checked': 20, 'total_series_with_anomalies': 10})```
```